In [ ]:
import numpy as np
import time
from deephaven import Server
from deephaven.plot.figure import Figure
from deephaven.ipc import table_pb2
from pydeephaven import Session

session = Session()

# Создайте временную таблицу с теми же данными, что и в массиве arr
t = table(table_pb2.TableDefinition(
    columns=[
        table_pb2.ColumnDescription(name="x", type=table_pb2.INT),
        table_pb2.ColumnDescription(name="y", type=table_pb2.DOUBLE)
    ]
))

# Создайте и отобразите фигуру
f = Figure().plot_xy(series_name="Figure", t=t, x="x", y="y").show()

# Подпишитесь на обновления временной таблицы
def update_chart(t):
    f.update_xy(t)

t.subscribe(update_chart)

# Запустите асинхронную функцию для генерации синусоиды
async def generate_sinusoid():
    arr = []
    while True:
        arr.append(np.sin(time.time()))
        t.update({"x": range(len(arr)), "y": arr})
        await asyncio.sleep(0.2)


# Запустите цикл событий асинхронно
asyncio.create_task(generate_sinusoid())
asyncio.get_event_loop().run_forever()